<a href="https://colab.research.google.com/github/MarceCorreal2/Robots-NT/blob/main/Procesamiento_Indicadores_Backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Este cuaderno limpia y organiza los datos del resumen del strategy analyzer de los robots y debe incluir los indicadores en la
# tabla robots y procesar la calificación

## Procesamiento de Indicadores de Backtest

El objetivo de este cuaderno es tomar los datos crudos de los resúmenes del *strategy analyzer* de tus robots, limpiarlos y extraer los indicadores clave mencionados. Una vez procesados, los datos se guardarán en un formato limpio para futuros análisis.

In [ ]:
# Celda 1 — Importes y configuración

import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [ ]:
# Celda 2 — Conectar Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Celda 3 — Definir Variables

bot_name = 'RB001_MNQ_A'
test_number = 'T001' # <--- Modifica este valor para diferentes pruebas (ej. 'T002')
instrument = 'MNQ'

print(f"Bot Name: {bot_name}")
print(f"Test Number: {test_number}")
print(f"Instrument: {instrument}")

Bot Name: RB001_MNQ_A
Test Number: T001
Instrument: MNQ


In [ ]:
# Celda 4 — Definir Rutas dinámicas

RAW_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/'
CLEAN_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/'

print(f"Ruta de Datos Crudos: {RAW_DATA_PATH}")
print(f"Ruta de Datos Limpios: {CLEAN_DATA_PATH}")

Ruta de Datos Crudos: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/
Ruta de Datos Limpios: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/


In [ ]:
# Celda 5 - Cargar y mostrar un archivo CSV de ejemplo con parsing robusto

import os
import pandas as pd

# Construir el nombre del archivo dinámicamente usando las variables definidas en Celda 3
sample_file_name = f'SA_{bot_name}_{test_number}.csv'
sample_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

print(f"Cargando archivo de ejemplo: {sample_file_path}")

try:
    # Read the first few lines to understand the structure and find the actual data start
    raw_lines = []
    with open(sample_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to cover potential header length
            line = f.readline()
            if not line: # EOF
                break
            raw_lines.append(line.strip())

    print("\nPrimeras 20 líneas del archivo crudo para inspección:")
    for i, line in enumerate(raw_lines[:20]):
        print(f"Línea {i+1}: {line}")

    # Try to find the line that indicates the start of the actual performance metrics
    # Common indicators like 'Total net profit' usually appear at the start of data section.
    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row != -1:
        print(f"\nIdentificado el inicio de los datos de indicadores en la línea (0-index): {data_start_row}")
        # Read the CSV again, skipping lines up to the identified data start
        # We set header=None because the first column will contain the indicator names,
        # and the subsequent columns are values (e.g., All trades, Long trades, Short trades).
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # Assuming the first column is the indicator name and the next are its values
        # We need to clean up the column names based on the context.
        # Let's just display the raw parsed DataFrame for now.
        print("\nDataFrame de indicadores procesado (primeras 5 filas):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del DataFrame procesado:")
        print(sample_df.columns.tolist())
    else:
        print("\nNo se pudo identificar el inicio de los datos de indicadores ('Total net profit' no encontrado). Se muestra la lectura inicial sin procesar.")
        # Fallback if specific data start not found, try reading with just separator
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';')
        print("\nPrimeras 5 filas del archivo de ejemplo (lectura básica):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del archivo de ejemplo (lectura básica):")
        print(sample_df.columns.tolist())

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al leer el archivo CSV: {e}")

Cargando archivo de ejemplo: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/SA_RB001_MNQ_A_T001.csv

Primeras 20 líneas del archivo crudo para inspección:
Línea 1: Performance;All trades;Long trades;Short trades;
Línea 2: Total net profit;$ 213,00;$ 213,00;$ 0,00;
Línea 3: Gross profit;$ 2243,00;$ 2243,00;$ 0,00;
Línea 4: Gross loss;-$ 2030,00;-$ 2030,00;$ 0,00;
Línea 5: Commission;$ 95,00;$ 95,00;$ 0,00;
Línea 6: Profit factor;1,10;1,10;1,00;
Línea 7: Max drawdown;-$ 995,60;-$ 995,60;$ 0,00;
Línea 8: Sharpe ratio;2,29;2,29;1,00;
Línea 9: Sortino ratio;1,00;1,00;1,00;
Línea 10: Ulcer index;0,01;0,01;0,00;
Línea 11: R squared;0,18;0,18;0,00;
Línea 12: Total Fees;$ 0,00;$ 0,00;$ 0,00;
Línea 13: Probability;42,13 %;42,13 %;0,00 %;
Línea 14: ;;;;
Línea 15: Start date;23/05/2026;;;
Línea 16: Start time;12:00 AM;;;
Línea 17: End date;3/06/2026;;;
Línea 18: End time;12:00 AM;;;
Línea 19: ;;;;
Línea 20: Total # of trades;50;50;0;

Identificado el inicio de los datos

In [ ]:
#Celda 6 - Limpieza de datos


import os
import pandas as pd
import re
from datetime import datetime

# Lista para almacenar los DataFrames de indicadores de cada archivo
all_indicators_list = []

# Función para limpiar y convertir valores numéricos
def clean_numeric_value(value):
    if isinstance(value, str):
        value = value.replace('$', '').replace(' ', '').replace('%', '').replace(',', '.')
        if value == '' or value == '-':
            return None
        try:
            return float(value)
        except ValueError:
            return value
    return value

# Usar el sample_file_name y sample_file_path ya definidos en Celda 5
# para procesar solo el archivo deseado.
print(f"Procesando el archivo especificado: {sample_file_name}")

# full_file_path ya está definido en Celda 5 y es el que queremos procesar
# Construimos full_file_path nuevamente aquí para asegurar que sea el correcto
# si Celda 5 no se ejecuta justo antes.
full_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

try:
    # --- Parsing robusto (similar a Celda 5) ---
    raw_lines = []
    with open(full_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to find data start
            line = f.readline()
            if not line: break
            raw_lines.append(line.strip())

    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row == -1:
        print(f"Advertencia: No se encontró el inicio de datos para {sample_file_name}. No se procesará este archivo.")
    else:
        temp_df = pd.read_csv(full_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # --- Limpieza y extracción (similar a Celda 6)---
        # Aplicar nombres de columna iniciales y establecer índice
        temp_df.columns = ['Performance', 'All trades', 'Long trades', 'Short trades', 'Extra_Column']
        temp_df = temp_df.drop(columns=['Extra_Column'])
        temp_df['Performance'] = temp_df['Performance'].str.strip()
        temp_df = temp_df.set_index('Performance')

        # Aplicar limpieza a valores numéricos
        for col in ['All trades', 'Long trades', 'Short trades']:
            temp_df[col] = temp_df[col].apply(clean_numeric_value)

        def get_indicator_value(df, indicator_name):
            try:
                return df.loc[indicator_name.strip(), 'All trades']
            except KeyError:
                return None

        # Extraer los indicadores solicitados
        NetProfit = get_indicator_value(temp_df, 'Total net profit')
        PF = get_indicator_value(temp_df, 'Profit factor')
        WR_probability = get_indicator_value(temp_df, 'Probability')
        DD_max = get_indicator_value(temp_df, 'Max drawdown')
        # RecoveryFactor = get_indicator_value(temp_df, 'Recovery factor') # Original line, now modified
        TotalTrades = get_indicator_value(temp_df, 'Total # of trades')
        Winners = get_indicator_value(temp_df, 'Number of winning trades')
        GrossProfit = get_indicator_value(temp_df, 'Gross profit')
        GrossLoss = get_indicator_value(temp_df, 'Gross loss')
        AvgWinTrade = get_indicator_value(temp_df, 'Avg winning trade') # New indicator
        AvgLossTrade = get_indicator_value(temp_df, 'Avg losing trade') # New indicator

        WR = WR_probability if WR_probability is not None else \
             (Winners / TotalTrades) * 100 if TotalTrades and Winners is not None and TotalTrades != 0 else None

        # Corrected PayoffRatio calculation
        PayoffRatio = (AvgWinTrade / abs(AvgLossTrade)) if AvgWinTrade and AvgLossTrade and AvgLossTrade != 0 else None

        # Calculate RecoveryFactor as Net Profit / abs(Max Drawdown)
        RecoveryFactor = (NetProfit / abs(DD_max)) if NetProfit is not None and DD_max is not None and DD_max != 0 else None

        # Extracción de fechas y cálculo de Net Profit/Mes
        FechaInicio = None
        FechaFin = None
        for line in raw_lines:
            if 'Start date' in line:
                match = re.search(r'Start date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaInicio = datetime.strptime(match.group(1), '%d/%m/%Y')
            elif 'End date' in line:
                match = re.search(r'End date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaFin = datetime.strptime(match.group(1), '%d/%m/%Y')

        NumMonths = None
        NetProfitPerMonth = None
        if NetProfit is not None and FechaInicio is not None and FechaFin is not None:
            delta = FechaFin - FechaInicio
            if delta.days > 0:
                NumMonths = delta.days / 30.44
                if NumMonths > 0:
                    NetProfitPerMonth = NetProfit / NumMonths

        # Extraer Robot Base, Test ID e Instrumento del nombre del archivo
        # Ejemplo: SA-RB01-MNQ-A-T001.csv
        file_parts = sample_file_name.replace('.csv', '').split('_') # Changed to split by underscore
        robot_base = file_parts[1] if len(file_parts) > 1 else None
        test_id = file_parts[-1] if len(file_parts) > 0 else None # Assuming last part is Test ID
        instrument_from_file = file_parts[2] if len(file_parts) > 2 else None

        # Crear un diccionario con los indicadores para este archivo
        file_indicators = {
            'Archivo': sample_file_name,
            'Robot Base': robot_base,
            'Test ID': test_id,
            'Instrumento': instrument_from_file,
            'NetProfit': NetProfit,
            'PF': PF,
            'WR': WR,
            'DD max': DD_max,
            'Recovery Factor': RecoveryFactor,
            'PayoffRatio': PayoffRatio,
            '# Trades': TotalTrades,
            '# Meses': NumMonths,
            'Net Profit/Mes': NetProfitPerMonth,
            'Fecha-Inicio': FechaInicio.strftime('%Y-%m-%d') if FechaInicio else None,
            'Fecha-Fin': FechaFin.strftime('%Y-%m-%d') if FechaFin else None,
            'Avg Win': AvgWinTrade,
            'Avg Loss': AvgLossTrade
        }
        all_indicators_list.append(file_indicators)

        # --- Nuevo código para guardar el archivo limpio individual sin subdirectorios dinámicos ---
        individual_df = pd.DataFrame([file_indicators])
        base_file_name = os.path.splitext(sample_file_name)[0] # e.g., 'SA-RB01-MNQ-A-T001'
        cleaned_individual_file_name = f"{base_file_name}-Limpio.csv"

        # La carpeta de salida es directamente CLEAN_DATA_PATH
        dynamic_output_dir = CLEAN_DATA_PATH
        os.makedirs(dynamic_output_dir, exist_ok=True)

        individual_output_path = os.path.join(dynamic_output_dir, cleaned_individual_file_name)
        individual_df.to_csv(individual_output_path, index=False)
        print(f"Indicadores individuales guardados en: {individual_output_path}")
        # --- Fin del nuevo código ---

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al procesar el archivo {sample_file_name}: {e}")

# Convertir la lista de diccionarios a un DataFrame consolidado
if all_indicators_list:
    consolidated_df = pd.DataFrame(all_indicators_list)
    print("\n--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---")
    display(consolidated_df.head())

    # No guardar el DataFrame consolidado aquí ya que la Celda 9 se encarga de la tabla maestra.
else:
    print("No se pudieron procesar indicadores de ningún archivo.")

Procesando el archivo especificado: SA_RB001_MNQ_A_T001.csv
Indicadores individuales guardados en: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/SA_RB001_MNQ_A_T001-Limpio.csv

--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---


,Archivo,Robot Base,Test ID,Instrumento,NetProfit,PF,WR,DD max,Recovery Factor,PayoffRatio,# Trades,# Meses,Net Profit/Mes,Fecha-Inicio,Fecha-Fin,Avg Win,Avg Loss
0,SA_RB001_MNQ_A_T001.csv,RB001,T001,MNQ,213.0,1.1,42.13,-995.6,0.213941,4.419704,50.0,0.361367,589.429091,2026-05-23,2026-06-03,224.3,-50.75


In [ ]:
# Celda 7 - Limpieza y reordenamiento de datos finales

import pandas as pd
import os

# Check if consolidated_df exists. If not, try to reconstruct it from the last saved individual cleaned file.
if 'consolidated_df' not in locals() and 'consolidated_df' not in globals():
    print("Advertencia: 'consolidated_df' no definido en el estado actual del kernel. Intentando cargar el último archivo limpio individual.")
    try:
        # Assuming bot_name, test_number, and CLEAN_DATA_PATH are defined in previous cells and are accessible.
        cleaned_individual_file_name = f"SA_{bot_name}_{test_number}-Limpio.csv"
        individual_output_path = os.path.join(CLEAN_DATA_PATH, cleaned_individual_file_name)

        if os.path.exists(individual_output_path):
            consolidated_df = pd.read_csv(individual_output_path)
            print(f"Éxito: 'consolidated_df' cargado desde {individual_output_path}.")
        else:
            print(f"Error: No se pudo cargar 'consolidated_df'. El archivo '{individual_output_path}' no existe. Por favor, asegúrese de ejecutar la 'Celda 6' primero.")
            consolidated_df = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
    except NameError as e:
        print(f"Error de variable al intentar cargar 'consolidated_df': {e}. Asegúrese de que 'bot_name', 'test_number' y 'CLEAN_DATA_PATH' estén definidos en celdas anteriores.")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort
    except Exception as e:
        print(f"Error inesperado al cargar 'consolidated_df': {e}")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort


# Proceed only if consolidated_df is not empty after the potential loading attempt
if not consolidated_df.empty:
    # Renombrar columnas para que coincidan exactamente con la solicitud del usuario
    # Make a copy to avoid SettingWithCopyWarning later, especially during column renames
    consolidated_df = consolidated_df.copy().rename(columns={
        'Fecha-Inicio': 'Fecha Inicio',
        'Fecha-Fin': 'Fecha Fin',
        'DD max': 'DD Max',
        'Net Profit/Mes': 'Net Profit/Mes'
    })

    # Definir el orden de las columnas solicitado por el usuario, incluyendo 'Robot Base' y 'Test ID'
    column_order = [
        'Fecha Inicio',
        'Fecha Fin',
        'Instrumento',
        'Robot Base', # Agregado para identificación única
        'Test ID',    # Agregado para identificación única
        '# Meses',
        '# Trades',
        'NetProfit',
        'Net Profit/Mes',
        'PF',
        'WR',
        'DD Max',
        'Recovery Factor',
        'PayoffRatio',
        'Avg Win',
        'Avg Loss'
    ]

    # Seleccionar y reordenar las columnas del DataFrame
    # Filter column_order to only include columns actually present in consolidated_df
    actual_columns_in_order = [col for col in column_order if col in consolidated_df.columns]
    final_df = consolidated_df[actual_columns_in_order]

    print("\n--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---")
    display(final_df.head())
else:
    final_df = pd.DataFrame() # Ensure final_df is defined even if consolidated_df is empty
    print("No se pudo generar 'final_df' porque 'consolidated_df' está vacío o no se pudo cargar.")


--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---


,Fecha Inicio,Fecha Fin,Instrumento,Robot Base,Test ID,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,2026-05-23,2026-06-03,MNQ,RB001,T001,0.361367,50.0,213.0,589.429091,1.1,42.13,-995.6,0.213941,4.419704,224.3,-50.75


In [ ]:
# Celda 8 - Verificar tipos de datos del DataFrame final
print("\n--- Tipos de datos del DataFrame final: ---")
display(final_df.dtypes)


--- Tipos de datos del DataFrame final: ---


,0
Fecha Inicio,object
Fecha Fin,object
Instrumento,object
Robot Base,object
Test ID,object
# Meses,float64
# Trades,float64
NetProfit,float64
Net Profit/Mes,float64
PF,float64


In [ ]:
# Celda 9  - Ingesta datos en Tabla Maestra

import os
import pandas as pd
import re
from datetime import datetime

MASTER_TABLE_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv'

# Helper function to extract Robot and Test ID from a string, designed to handle various formats
def parse_robot_test_id(s):
    if pd.isna(s):
        return None, None
    s = str(s).strip()

    robot_extracted = None
    test_id_extracted = None

    # Try to extract Robot (e.g., RB001)
    match_robot = re.search(r'RB(\d+)', s, re.IGNORECASE)
    if match_robot:
        robot_extracted = f"RB{match_robot.group(1).zfill(3)}" # Ensure 3 digits, e.g., RB001

    # Try to extract Test ID (e.g., T001)
    match_test_id = re.search(r'T(\d+)', s, re.IGNORECASE)
    if match_test_id:
        test_id_extracted = f"T{match_test_id.group(1).zfill(3)}" # Ensure 3 digits, e.g., T001

    return robot_extracted, test_id_extracted


# Renombrar columnas en final_df antes de cualquier otra operación para asegurar consistencia
# Usamos .copy() para evitar SettingWithCopyWarning
final_df_to_add = final_df.rename(columns={'Robot Base': 'Robot', 'Test ID': 'Numero del Test'}).copy()

# Asegurar que las columnas clave sean de tipo string para la comparación
final_df_to_add['Robot'] = final_df_to_add['Robot'].astype(str)
final_df_to_add['Numero del Test'] = final_df_to_add['Numero del Test'].astype(str)

# Convertir las columnas de fecha en final_df_to_add a tipo datetime para compatibilidad
for col in ['Fecha Inicio', 'Fecha Fin']:
    if col in final_df_to_add.columns:
        final_df_to_add.loc[:, col] = pd.to_datetime(final_df_to_add[col], errors='coerce')


# Verificar si el archivo de la tabla maestra existe
if os.path.exists(MASTER_TABLE_PATH):
    print(f"Cargando tabla maestra desde: {MASTER_TABLE_PATH}")
    master_df = pd.read_csv(MASTER_TABLE_PATH)

    # --- START: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---
    # First, ensure 'Robot' and 'Numero del Test' columns exist, possibly from old names
    if 'Robot Base' in master_df.columns and 'Robot' not in master_df.columns:
        master_df = master_df.rename(columns={'Robot Base': 'Robot'})
    if 'Test ID' in master_df.columns and 'Numero del Test' not in master_df.columns:
        master_df = master_df.rename(columns={'Test ID': 'Numero del Test'})

    # If the columns still don't exist, create them with placeholder
    if 'Robot' not in master_df.columns:
        master_df['Robot'] = pd.NA
    if 'Numero del Test' not in master_df.columns:
        master_df['Numero del Test'] = pd.NA

    # Apply robust parsing to standardize Robot and Numero del Test
    standardized_robot_col = []
    standardized_test_id_col = []

    for idx, row in master_df.iterrows():
        # Get current values, try to use existing if they seem valid
        current_robot_val = row['Robot']
        current_test_id_val = row['Numero del Test']

        # Attempt to parse from current 'Robot' value
        parsed_r_from_robot, parsed_t_from_robot = parse_robot_test_id(current_robot_val)
        # Attempt to parse from current 'Numero del Test' value
        parsed_r_from_test_id, parsed_t_from_test_id = parse_robot_test_id(current_test_id_val)
        # Attempt to parse from 'Archivo' column if it exists and current keys are problematic
        parsed_r_from_file = None
        parsed_t_from_file = None
        if 'Archivo' in master_df.columns and (pd.isna(current_robot_val) or pd.isna(current_test_id_val) or not str(current_robot_val).startswith('RB') or not str(current_test_id_val).startswith('T')):
             parsed_r_from_file, parsed_t_from_file = parse_robot_test_id(row['Archivo'])


        # Prioritize values that look correct (e.g., start with 'RB'/'T')
        # Combine parsed results, prioritizing from specific columns or more complete sources
        final_robot = None
        if parsed_r_from_robot and parsed_r_from_robot.startswith('RB'):
            final_robot = parsed_r_from_robot
        elif parsed_r_from_test_id and parsed_r_from_test_id.startswith('RB'):
            final_robot = parsed_r_from_test_id
        elif parsed_r_from_file and parsed_r_from_file.startswith('RB'):
            final_robot = parsed_r_from_file

        final_test_id = None
        if parsed_t_from_test_id and parsed_t_from_test_id.startswith('T'):
            final_test_id = parsed_t_from_test_id
        elif parsed_t_from_robot and parsed_t_from_robot.startswith('T'):
            final_test_id = parsed_t_from_robot
        elif parsed_t_from_file and parsed_t_from_file.startswith('T'):
            final_test_id = parsed_t_from_file

        standardized_robot_col.append(final_robot if final_robot else 'UNKNOWN_ROBOT')
        standardized_test_id_col.append(final_test_id if final_test_id else 'UNKNOWN_TEST')

    master_df['Robot'] = standardized_robot_col
    master_df['Numero del Test'] = standardized_test_id_col

    # Ensure 'Robot' and 'Numero del Test' are string type after standardization
    master_df['Robot'] = master_df['Robot'].astype(str)
    master_df['Numero del Test'] = master_df['Numero del Test'].astype(str)
    # --- END: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---

    # --- START: Filter out non-identifiable rows from the loaded master_df ---
    initial_rows = len(master_df)
    master_df = master_df[
        (master_df['Robot'] != 'UNKNOWN_ROBOT') &
        (master_df['Numero del Test'] != 'UNKNOWN_TEST')
    ].copy() # Use .copy() to avoid SettingWithCopyWarning
    if len(master_df) < initial_rows:
        print(f"Advertencia: Se eliminaron {initial_rows - len(master_df)} filas no identificables (UNKNOWN_ROBOT/UNKNOWN_TEST) de la tabla maestra cargada.")
    # --- END: Filter out non-identifiable rows ---

    # --- ADDED: Clean up potentially erroneous columns from loaded master_df BEFORE merging ---
    # Drop 'NetProfit/Mes' (without space) if 'Net Profit/Mes' (with space) exists, as the latter is correct.
    if 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' in master_df.columns:
        print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) de la tabla maestra cargada.")
        master_df = master_df.drop(columns=['NetProfit/Mes'])
    # Rename 'NetProfit/Mes' (without space) to 'Net Profit/Mes' (with space) if only the former exists.
    elif 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' not in master_df.columns:
        print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en la tabla maestra cargada.")
        master_df = master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})

    # Drop 'Calificación Final' if it exists in the loaded master_df, as it will be recalculated later.
    if 'Calificación Final' in master_df.columns:
        print("Eliminando columna 'Calificación Final' de la tabla maestra cargada para recalcularla.")
        master_df = master_df.drop(columns=['Calificación Final'])
    # --- END ADDED CLEANUP ---

    # Convertir las columnas de fecha en master_df a tipo datetime si es necesario
    for col in ['Fecha Inicio', 'Fecha Fin']:
        if col in master_df.columns:
            # Convert to string first to avoid errors with mixed types, then to datetime
            master_df[col] = master_df[col].astype(str)
            master_df.loc[:, col] = pd.to_datetime(master_df[col], errors='coerce')


    print("\n--- Tabla Maestra Original (primeras 5 filas estandarizadas y limpiadas): ---")
    display(master_df.head())

    # Get all unique (Robot, Numero del Test) pairs from the current processed data
    processed_keys = final_df_to_add[['Robot', 'Numero del Test']].drop_duplicates()

    # Create a boolean mask to identify rows in master_df that should be removed
    # These are rows whose (Robot, Numero del Test) pair is present in processed_keys
    mask_to_remove = master_df.set_index(['Robot', 'Numero del Test']).index.isin(
        processed_keys.set_index(['Robot', 'Numero del Test']).index
    )

    # Filter master_df to remove rows that are present in the current processed data
    master_df_filtered = master_df[~mask_to_remove].copy()

    # Concatenate the filtered master_df with the new records
    updated_master_df = pd.concat([master_df_filtered, final_df_to_add], ignore_index=True)

else:
    print(f"Advertencia: La tabla maestra no existe en {MASTER_TABLE_PATH}. Creando una nueva tabla maestra con los datos actuales.")
    updated_master_df = final_df_to_add.copy() # La primera entrada será la ejecución actual

# --- NUEVA LÓGICA DE LIMPIEZA DE COLUMNAS SIMILARES (retained for final check) ---
# This block acts as a fallback to ensure that after concatenation,
# if 'NetProfit/Mes' (without space) is still present and 'Net Profit/Mes' (with space) also exists,
# the former is removed. This might catch issues if final_df_to_add somehow introduced it, though unlikely.
if 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' in updated_master_df.columns:
    print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) duplicada en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.drop(columns=['NetProfit/Mes'])
elif 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' not in updated_master_df.columns:
    print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})
# --- FIN NUEVA LÓGICA DE LIMPIEZA ---


# Reordenar columnas para colocar 'Robot' y 'Numero del Test' al principio
expected_cols_order = [
    'Robot', 'Numero del Test', 'Fecha Inicio', 'Fecha Fin', 'Instrumento',
    '# Meses', '# Trades', 'NetProfit', 'Net Profit/Mes', 'PF', 'WR', # Asegurar que el nombre aquí sea el correcto
    'DD Max', 'Recovery Factor', 'PayoffRatio', 'Avg Win', 'Avg Loss'
]
# Add any missing columns to updated_master_df that are in expected_cols_order, filling with NaN
for col in expected_cols_order:
    if col not in updated_master_df.columns:
        updated_master_df[col] = pd.NA

# Reorder columns based on expected_cols_order
updated_master_df = updated_master_df[expected_cols_order]


print("\n--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---")
display(updated_master_df.head())

# Guardar la tabla maestra actualizada
updated_master_df.to_csv(MASTER_TABLE_PATH, index=False)
print(f"\nTabla maestra actualizada y guardada en: {MASTER_TABLE_PATH}")

Advertencia: La tabla maestra no existe en /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv. Creando una nueva tabla maestra con los datos actuales.

--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,RB001,T001,2026-05-23 00:00:00,2026-06-03 00:00:00,MNQ,0.361367,50.0,213.0,589.429091,1.1,42.13,-995.6,0.213941,4.419704,224.3,-50.75



Tabla maestra actualizada y guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv


In [ ]:
# Celda 10 - Verificar datos

MASTER_TABLE_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv'

print(f"Verificando el contenido de: {MASTER_TABLE_PATH}")
if os.path.exists(MASTER_TABLE_PATH):
    verified_master_df = pd.read_csv(MASTER_TABLE_PATH)
    print("\n--- Contenido del archivo de la Tabla Maestra (primeras 5 filas): ---")
    display(verified_master_df.head())
    print(f"Total de filas en la tabla maestra: {len(verified_master_df)}")
else:
    print("El archivo de la Tabla Maestra no se encontró.")

Verificando el contenido de: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots.csv

--- Contenido del archivo de la Tabla Maestra (primeras 5 filas): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,# Trades,NetProfit,Net Profit/Mes,PF,WR,DD Max,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,RB001,T001,2026-05-23 00:00:00,2026-06-03 00:00:00,MNQ,0.361367,50.0,213.0,589.429091,1.1,42.13,-995.6,0.213941,4.419704,224.3,-50.75


Total de filas en la tabla maestra: 1


---